# Bài tập về nhà — Bài TH 02 (Web3.py / Ape / Geth)

Theo mục **4. Bài tập về nhà** trong `Bai02.pdf`.

| Bài | File chính | Nội dung |
|-----|------------|----------|
| 1 | `bai1_blockchain_health.py` | Sức khỏe blockchain (block, hash, gas) |
| 2 | `contracts/DebtBook.vy` + `scripts/bai2_debtbook.py` | Sổ nợ cá nhân |
| 3 | `bai3_low_gas_tx.py` | Tx gas thấp → quan sát lỗi |
| 4 | `contracts/Voting.vy` + `scripts/bai4_voting.py` | Bỏ phiếu phi tập trung |
| 5 | `contracts/SupplyChain.vy` + `scripts/bai5_supply_chain.py` | Chuỗi cung ứng nông sản |

**Môi trường (Windows):**
- Geth: `D:\Tools\Geth\geth.exe --dev --http ... --ipcdisable` (port **8545**)
- Python/Ape: `.venv-ape`
- CWD khi chạy Ape: thư mục này `BTTH2_VeNha` (hoặc copy contract vào project Ape `TH/w2`)


## Setup nhanh

```powershell
cd D:\Study\IS355\TH\w2\BTTH2_VeNha
# Geth (terminal rieng, giu chay):
D:\Tools\Geth\geth.exe --dev --http --http.api eth,web3,net --http.addr 127.0.0.1 --http.port 8545 --ipcdisable
```

Kernel notebook: **`.venv-ape`** (có `web3`, `eth-ape`, `vyper`).


In [1]:
from pathlib import Path
from vyper.compiler import compile_code

def compile_vy(filename: str):
    src = Path(filename).read_text(encoding="utf-8")
    print("File:", Path(filename).resolve())
    print("-" * 50)
    try:
        result = compile_code(src, output_formats=["abi", "bytecode"])
        abi = result.get("abi") or []
        fns = [x.get("name") for x in abi if isinstance(x, dict) and x.get("type") == "function"]
        bytecode = str(result.get("bytecode") or "")
        print("[OK] Bien dich thanh cong")
        print("ABI functions:", ", ".join(fns) if fns else "(khong co)")
        print("Bytecode (64 ky tu dau):", bytecode[:64], ("..." if len(bytecode) > 64 else ""))
    except Exception as e:
        print("[FAIL]", type(e).__name__)
        print(e)


---
## Bài 1 — Kiểm tra “Sức khỏe” Blockchain

**Yêu cầu:** Kết nối Geth, in:
1. Số block hiện tại  
2. Block mới nhất: hash, miner, số giao dịch  
3. Gas trung bình của block đó  

Gợi ý: `w3.eth.block_number`, `w3.eth.get_block()`.


In [2]:
# Chay khi Geth dang mo o :8545
# Hoac: D:\Study\IS355\.venv-ape\Scripts\python.exe bai1_blockchain_health.py
from web3 import Web3

w3 = Web3(Web3.HTTPProvider("http://127.0.0.1:8545"))
assert w3.is_connected(), "Mo Geth truoc"

block_number = w3.eth.block_number
block = w3.eth.get_block(block_number)

print("1) So block hien tai :", block_number)
print("2) Block moi nhat:")
print("   hash  :", block["hash"].hex())
print("   miner :", block.get("miner") or block.get("author"))
txs = block.get("transactions") or []
print("   so tx :", len(txs))
gas_used = block.get("gasUsed") or 0
avg = (gas_used / len(txs)) if txs else 0
print("3) Gas trung binh/tx :", avg, "(gasUsed=", gas_used, ")")


1) So block hien tai : 0
2) Block moi nhat:
   hash  : 790ac206f4e9d3837e8284a6b81d69d832d6974d10735757ccad8ad35086d6ad
   miner : 0x0000000000000000000000000000000000000000
   so tx : 0
3) Gas trung binh/tx : 0 (gasUsed= 0 )


---
## Bài 2 — Sổ Nợ Cá Nhân (`DebtBook.vy`)

**Yêu cầu:**
- `addDebt(debtor, amount)` — chỉ **owner**
- `getDebt(debtor) -> uint256`
- Deploy Geth + kiểm thử Ape


In [3]:
compile_vy("contracts/DebtBook.vy")

File: D:\Study\IS355\TH\w2\BTTH2_VeNha\contracts\DebtBook.vy
--------------------------------------------------
[OK] Bien dich thanh cong
ABI functions: addDebt, getDebt, owner
Bytecode (64 ky tu dau): 0x3461001957335f556101b561001d610000396101b5610000f35b5f80fd5f35 ...


In [4]:
print(Path("contracts/DebtBook.vy").read_text(encoding="utf-8"))

# @version ^0.4.3

"""Sổ nợ cá nhân: owner ghi nợ cho từng địa chỉ."""

owner: public(address)
debts: HashMap[address, uint256]

@deploy
def __init__():
    self.owner = msg.sender

@external
def addDebt(debtor: address, amount: uint256):
    """Chi owner duoc them no."""
    assert msg.sender == self.owner, "only owner"
    assert debtor != empty(address), "bad debtor"
    self.debts[debtor] += amount

@external
@view
def getDebt(debtor: address) -> uint256:
    return self.debts[debtor]



### Chạy trên Ape (PowerShell)

Đặt project Ape trỏ tới folder này (có `contracts/`), hoặc copy `DebtBook.vy` vào `TH/w2/contracts`.

```powershell
cd D:\Study\IS355\TH\w2\BTTH2_VeNha
D:\Study\IS355\.venv-ape\Scripts\ape.exe compile
D:\Study\IS355\.venv-ape\Scripts\ape.exe test
D:\Study\IS355\.venv-ape\Scripts\ape.exe run bai2_debtbook --network ethereum:local:http://127.0.0.1:8545
```

File test: `tests/test_debtbook.py` · Script: `scripts/bai2_debtbook.py`


---
## Bài 3 — Giao dịch thất bại (gas quá thấp)

**Yêu cầu:** Gửi ETH với `gas` cố tình thấp → quan sát lỗi Geth → giải thích + cách sửa.


In [5]:
# gas=100 << 21000 -> intrinsic gas too low / reject
from web3 import Web3

w3 = Web3(Web3.HTTPProvider("http://127.0.0.1:8545"))
assert w3.is_connected()

sender = w3.eth.accounts[0]
recipient = sender
tx = {
    "from": sender,
    "to": recipient,
    "value": Web3.to_wei(0.001, "ether"),
    "gas": 100,
    "gasPrice": Web3.to_wei("1", "gwei"),
    "nonce": w3.eth.get_transaction_count(sender),
}
print("Thu gui gas=100 ...")
try:
    h = w3.eth.send_transaction(tx)
    print("hash", h.hex())
    r = w3.eth.wait_for_transaction_receipt(h)
    print("status", r["status"])
except Exception as e:
    print("LOI:", type(e).__name__, e)

print('''
Giai thich:
- Transfer can intrinsic gas ~21000.
- gas=100 qua thap -> node tu choi (intrinsic gas too low) hoac tx fail.
Cach sua: gas=21000 (transfer) / uoc luong dung neu goi contract.
''')


Thu gui gas=100 ...
LOI: Web3RPCError {'code': -32000, 'message': 'intrinsic gas too low: gas 100, minimum needed 21000'}

Giai thich:
- Transfer can intrinsic gas ~21000.
- gas=100 qua thap -> node tu choi (intrinsic gas too low) hoac tx fail.
Cach sua: gas=21000 (transfer) / uoc luong dung neu goi contract.



---
## Bài 4 — Bỏ phiếu phi tập trung (`Voting.vy`)

**Yêu cầu:**
- Owner thêm ứng viên (`addCandidate`)
- Mỗi account vote **1 lần** (`vote(candidate_id)`)
- `getWinner()` → (id, tên, số phiếu)
- Deploy Geth + script 3 account bỏ phiếu


In [6]:
compile_vy("contracts/Voting.vy")

File: D:\Study\IS355\TH\w2\BTTH2_VeNha\contracts\Voting.vy
--------------------------------------------------
[OK] Bien dich thanh cong
ABI functions: addCandidate, vote, getVotes, getWinner, owner, candidate_count
Bytecode (64 ky tu dau): 0x3461001957335f556105ee61001d610000396105ee610000f35b5f80fd5f35 ...


### Chạy script bỏ phiếu

```powershell
cd D:\Study\IS355\TH\w2\BTTH2_VeNha
D:\Study\IS355\.venv-ape\Scripts\ape.exe compile
D:\Study\IS355\.venv-ape\Scripts\ape.exe run bai4_voting --network ethereum:local:http://127.0.0.1:8545
```

Script `scripts/bai4_voting.py`: 3 voter chọn An/An/Bình → An thắng.


---
## Bài 5 — Chuỗi cung ứng nông sản (`SupplyChain.vy`)

**Bối cảnh:** Hợp tác xã rau sạch — lưu hành trình nông dân → kho → siêu thị trên blockchain (không sửa được).

**Yêu cầu contract:**
- Struct `Product`: id, name, currentOwner, status  
- `addProduct` (manager) · `transferProduct` · `getProduct`  
- Event `ProductUpdated`  
- Deploy Geth + script mô phỏng + query log event


In [7]:
compile_vy("contracts/SupplyChain.vy")

File: D:\Study\IS355\TH\w2\BTTH2_VeNha\contracts\SupplyChain.vy
--------------------------------------------------
[OK] Bien dich thanh cong
ABI functions: addProduct, transferProduct, getProduct, manager
Bytecode (64 ky tu dau): 0x3461001957335f5561087361001d61000039610873610000f35b5f80fd5f35 ...


### Chạy mô phỏng

```powershell
cd D:\Study\IS355\TH\w2\BTTH2_VeNha
D:\Study\IS355\.venv-ape\Scripts\ape.exe compile
D:\Study\IS355\.venv-ape\Scripts\ape.exe run bai5_supply_chain --network ethereum:local:http://127.0.0.1:8545
```

Script: thêm `"Rau cai sach"` id=1 → chuyển nông dân / kho lạnh / siêu thị → in `ProductUpdated` logs.


In [8]:
# Mo rong: query event bang Web3.py (sau khi da deploy, dien contract_address + ABI)
# Vi du doc event tu receipt / get_logs — xem them scripts/bai5_supply_chain.py (Ape query)
print("Dung ape run bai5_supply_chain de in ProductUpdated logs.")
print("Hoac Web3: contract.events.ProductUpdated.get_logs(fromBlock=0, toBlock='latest')")


Dung ape run bai5_supply_chain de in ProductUpdated logs.
Hoac Web3: contract.events.ProductUpdated.get_logs(fromBlock=0, toBlock='latest')
